In [3]:
import rasterio
import numpy as np

with rasterio.open('../data/processed/caldas/caldas_dem.tif') as src:
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Tipo de dato:", src.dtypes[0])
    print("Shape:", src.shape)
    
    dem = src.read(1)
    print("Valores únicos (muestra):", np.unique(dem)[:10])
    print("Rango de elevación:", np.nanmin(dem), "-", np.nanmax(dem))


CRS: EPSG:4326
Resolución: (0.00026949458522372663, 0.0002694945853054483)
Tipo de dato: int16
Shape: (3634, 4805)
Valores únicos (muestra): [  0 144 145 146 147 148 149 150 151 152]
Rango de elevación: 0 - 5288


In [ ]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

# Ruta del DEM original en grados
src_path = '../data/processed/caldas/caldas_dem.tif'
dst_path = '../data/processed/caldas/caldas_dem_9377.tif'

# CRS de salida: EPSG:9377 (MAGNA-SIRGAS Origen Nacional)
dst_crs = 'EPSG:9377'

with rasterio.open(src_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=30
    )
    
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height,
        'dtype': 'float32'  # Elevación como flotantes
    })

    with rasterio.open(dst_path, 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear
            )

print("DEM reproyectado a metros y guardado como:", dst_path)
